# Trabajo practico - Ejercicio 1 (a y b)

Resolucion de la representacion del subte y la caracterizacion de estaciones respetando el estilo del notebook de referencia, pero con rutas locales para que se pueda ejecutar dentro de este repo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
FIGURES_DIR = BASE_DIR / 'figures'
OUTPUT_DIR = BASE_DIR / 'outputs'
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
estaciones = pd.read_csv(DATA_DIR / 'estaciones.txt')
conexiones = pd.read_csv(DATA_DIR / 'conexiones.txt')
posiciones = pd.read_csv(DATA_DIR / 'estaciones_posicion.txt')
diagrama = pd.read_csv(DATA_DIR / 'estaciones_posicion_diagrama.txt')

estaciones.head()

In [ ]:
G = nx.from_pandas_edgelist(conexiones, source='source', target='target', edge_attr=True)
nx.set_node_attributes(G, estaciones.set_index('id').to_dict('index'))
nx.set_edge_attributes(G, {edge: 1 / G.edges[edge]['tiempo'] for edge in G.edges()}, name='invtiempo')

pos_geo = {row['id']: (float(row['long'].replace(',', '.')), float(row['lat'].replace(',', '.'))) for _, row in posiciones.iterrows()}
pos_diag = {row['id']: (-row['y'], row['x']) for _, row in diagrama.iterrows()}
colors_dict = {'A': 'tab:blue', 'B': 'tab:red', 'C': 'navy', 'D': 'tab:green', 'E': 'tab:purple', 'H': 'gold'}
colors = [colors_dict[G.nodes[node]['linea']] for node in G.nodes()]
labels = nx.get_node_attributes(G, 'nombre')

## 1.a Representacion del subte en el mapa

In [ ]:
plt.figure(figsize=(10, 10))
nx.draw_networkx_nodes(G, pos_geo, node_size=250, node_color=colors, edgecolors='black', linewidths=0.4)
nx.draw_networkx_edges(G, pos_geo, edge_color='gray', width=[G.edges[edge]['tiempo'] * 0.6 for edge in G.edges()])
nx.draw_networkx_labels(G, pos_geo, labels=labels, font_size=6, font_color='black')
plt.title('Ejercicio 1.a - Red de subte de CABA')
plt.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '1a_red_subte_mapa.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
nx.draw_networkx_nodes(G, pos_diag, node_size=250, node_color=colors, edgecolors='black', linewidths=0.4)
nx.draw_networkx_edges(G, pos_diag, edge_color='gray', width=[G.edges[edge]['tiempo'] * 0.6 for edge in G.edges()])
nx.draw_networkx_labels(G, pos_diag, labels=labels, font_size=6, font_color='black')
plt.title('Diagrama estilizado de la red')
plt.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '1a_red_subte_diagrama.png', dpi=200, bbox_inches='tight')
plt.show()

print(G.number_of_nodes())
print(G.number_of_edges())
print(sum(1 for edge in G.edges() if G.edges[edge]['tipo'] == 'transbordo'))

## 1.b Caracterizacion del rol de las estaciones

In [ ]:
degree = dict(G.degree())
betweenness = nx.betweenness_centrality(G, weight='tiempo')
closeness = nx.closeness_centrality(G, distance='tiempo')
eigenvector = nx.eigenvector_centrality(G, weight='invtiempo', max_iter=10000)
eccentricity = nx.eccentricity(G, weight='tiempo')

In [ ]:
def plot_measure(values, title, filename, scale, base=0):
    plt.figure(figsize=(10, 6))
    nx.draw_networkx_nodes(
        G,
        pos_geo,
        node_size=[base + values[node] * scale for node in G.nodes()],
        node_color=colors,
        edgecolors='black',
        linewidths=0.4,
    )
    nx.draw_networkx_edges(G, pos_geo, edge_color='gray', width=1.5)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
plot_measure({node: float(value) for node, value in degree.items()}, 'Cantidad de conexiones', '1b_grado.png', scale=90, base=120)
plot_measure(betweenness, 'Betweenness', '1b_betweenness.png', scale=7000, base=80)
plot_measure(closeness, 'Closeness', '1b_closeness.png', scale=5000, base=80)
plot_measure(eigenvector, 'Centralidad de autovector', '1b_autovector.png', scale=4000, base=80)
plot_measure(eccentricity, 'Excentricidad', '1b_excentricidad.png', scale=20, base=40)

In [ ]:
summary = pd.DataFrame({
    'id': list(G.nodes()),
    'estacion': [G.nodes[node]['nombre'] for node in G.nodes()],
    'linea': [G.nodes[node]['linea'] for node in G.nodes()],
    'grado': [degree[node] for node in G.nodes()],
    'betweenness': [betweenness[node] for node in G.nodes()],
    'closeness': [closeness[node] for node in G.nodes()],
    'autovector': [eigenvector[node] for node in G.nodes()],
    'excentricidad': [eccentricity[node] for node in G.nodes()],
})
summary['r_grado'] = summary['grado'].rank(ascending=False, method='min')
summary['r_betweenness'] = summary['betweenness'].rank(ascending=False, method='min')
summary['r_closeness'] = summary['closeness'].rank(ascending=False, method='min')
summary['r_autovector'] = summary['autovector'].rank(ascending=False, method='min')
summary['r_excentricidad'] = summary['excentricidad'].rank(ascending=True, method='min')
summary['score_central'] = summary[['r_grado', 'r_betweenness', 'r_closeness', 'r_autovector', 'r_excentricidad']].mean(axis=1)
summary['score_periferico'] = summary[['grado', 'betweenness', 'closeness', 'autovector']].rank(ascending=True, method='min').mean(axis=1) + summary['excentricidad'].rank(ascending=False, method='min') / 5
summary.sort_values(['score_central', 'betweenness', 'closeness'], ascending=[True, False, False]).head(10)

In [ ]:
summary.to_csv(OUTPUT_DIR / 'centralidad_estaciones.csv', index=False)
print(f"Radio de la red: {nx.radius(G, weight='tiempo')} min")
print(f"Diametro de la red: {nx.diameter(G, weight='tiempo')} min")
print('\nMas perifericas por excentricidad:')
display(summary.sort_values(['excentricidad', 'closeness'], ascending=[False, True])[['estacion', 'linea', 'excentricidad', 'closeness']].head(10))